<a href="https://colab.research.google.com/github/pramodjella/QuantumML-IIT-Delhi-/blob/main/ML_vs_QML_qiskit_implementaion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install qiskit qiskit-machine-learning qiskit-algorithms datasets scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.9/231.9 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.4/54.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 13.6 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [ ]:
"""
Quantum vs Classical Sentiment Analysis: Full Comparison & Visualization
(Corrected for NumPy Array and One-Hot Encoding issues)
"""

import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

# Qiskit Imports
from qiskit.circuit.library import ZZFeatureMap, RealAmplitudes
from qiskit_algorithms.optimizers import COBYLA
from qiskit_algorithms.utils import algorithm_globals
from qiskit_algorithms.state_fidelities import ComputeUncompute
from qiskit_machine_learning.algorithms import VQC, QSVC
from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit.primitives import StatevectorSampler

# Configuration
algorithm_globals.random_seed = 42
np.random.seed(42)
sns.set_theme(style="whitegrid")

class QuantumComparison:
    def __init__(self, n_features=4):
        self.n_features = n_features
        # Define dataset sizes to test
        self.data_sizes = {
            'Small': 400,
            'Medium': 800,
            # 'Large': 1200
        }
        self.results_df = pd.DataFrame()

    def load_data(self, n_samples):
        """
        Loads SST-2 and converts all outputs to NumPy arrays.
        """
        print(f"\n... Processing Data (N={n_samples}) ...")
        dataset = load_dataset('glue', 'sst2')

        # 1. Prepare Training Pool
        raw_train_texts = dataset['train']['sentence'][:n_samples]
        raw_train_labels = dataset['train']['label'][:n_samples]

        # 2. Split Training Pool -> Train (80%) + Validation (20%)
        # train_test_split handles lists but returns lists if input is list
        X_train_txt, X_val_txt, y_train, y_val = train_test_split(
            raw_train_texts, raw_train_labels, test_size=0.2, random_state=42
        )

        # 3. Prepare Test Set (From official validation set)
        X_test_txt = dataset['validation']['sentence'][:200]
        y_test = dataset['validation']['label'][:200]

        # 4. Vectorize (TF-IDF)
        vectorizer = TfidfVectorizer(max_features=100, stop_words='english')
        X_train_vec = vectorizer.fit_transform(X_train_txt).toarray()
        X_val_vec = vectorizer.transform(X_val_txt).toarray()
        X_test_vec = vectorizer.transform(X_test_txt).toarray()

        # 5. PCA Reduction
        pca = PCA(n_components=self.n_features)
        X_train_pca = pca.fit_transform(X_train_vec)
        X_val_pca = pca.transform(X_val_vec)
        X_test_pca = pca.transform(X_test_vec)

        # 6. Normalize to [0, 1] for Quantum Embedding
        def normalize(data, ref):
            return (data - ref.min()) / (ref.max() - ref.min() + 1e-8)

        X_train_q = normalize(X_train_pca, X_train_pca)
        X_val_q = normalize(X_val_pca, X_train_pca)
        X_test_q = normalize(X_test_pca, X_train_pca)

        # 7. *** CRITICAL FIX: Convert Labels to NumPy Arrays ***
        y_train = np.array(y_train)
        y_val = np.array(y_val)
        y_test = np.array(y_test)

        return (X_train_pca, y_train, X_val_pca, y_val, X_test_pca, y_test), \
               (X_train_q, y_train, X_val_q, y_val, X_test_q, y_test)

    def train_evaluate(self, model, name, size_name, data_pack, is_vqc=False):
        """
        Generic training function with VQC-specific handling
        """
        X_train, y_train, X_val, y_val, X_test, y_test = data_pack

        print(f"  Training {name}...", end=" ")
        start = time.time()

        # *** VQC Specific Handling ***
        if is_vqc:
            # VQC training often requires One-Hot Encoded targets
            encoder = OneHotEncoder(sparse_output=False)
            y_train_fit = encoder.fit_transform(y_train.reshape(-1, 1))
            # VQC fit
            model.fit(X_train, y_train_fit)
        else:
            # Classical/QSVC use 1D labels
            model.fit(X_train, y_train)

        duration = time.time() - start
        print(f"Done ({duration:.2f}s)")

        # Predictions
        # Note: VQC.predict usually returns class labels (1D) even if trained on One-Hot
        y_train_pred = model.predict(X_train)
        y_val_pred = model.predict(X_val)
        y_test_pred = model.predict(X_test)

        # Safety check: If VQC returns One-Hot predictions, convert to 1D
        if len(y_test_pred.shape) > 1 and y_test_pred.shape[1] > 1:
            y_train_pred = np.argmax(y_train_pred, axis=1)
            y_val_pred = np.argmax(y_val_pred, axis=1)
            y_test_pred = np.argmax(y_test_pred, axis=1)

        # Metrics
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_test, y_test_pred, average='weighted', zero_division=0
        )

        row = {
            'Data Size': size_name,
            'Model': name,
            'Train Samples': len(X_train),
            'Train Acc': accuracy_score(y_train, y_train_pred),
            'Val Acc': accuracy_score(y_val, y_val_pred),
            'Test Acc': accuracy_score(y_test, y_test_pred),
            'Precision': precision,
            'Recall': recall,
            'F1 Score': f1,
            'Time (s)': duration
        }
        return row

    def run(self):
        all_rows = []

        for size_name, n_samples in self.data_sizes.items():
            # Get Data
            c_data, q_data = self.load_data(n_samples)

            # --- Classical Models ---
            lr = LogisticRegression(max_iter=1000)
            all_rows.append(self.train_evaluate(lr, "Classical LR", size_name, c_data))

            svc = SVC(kernel='rbf')
            all_rows.append(self.train_evaluate(svc, "Classical SVC", size_name, c_data))

            # --- Quantum Setup ---
            feature_map = ZZFeatureMap(feature_dimension=self.n_features, reps=2, entanglement='linear')
            ansatz = RealAmplitudes(num_qubits=self.n_features, reps=3, entanglement='linear')
            sampler = StatevectorSampler(seed=42)

            # --- Quantum VQC ---
            optimizer = COBYLA(maxiter=80)
            # Note: We pass num_classes implicitly via the dataset, or explicitly if needed.
            # Usually VQC infers from One-Hot training data.
            vqc = VQC(sampler=sampler, feature_map=feature_map, ansatz=ansatz, optimizer=optimizer)

            # Pass is_vqc=True to handle One-Hot Encoding
            all_rows.append(self.train_evaluate(vqc, "Quantum VQC", size_name, q_data, is_vqc=True))

            # --- Quantum QSVC ---
            fidelity = ComputeUncompute(sampler=sampler)
            kernel = FidelityQuantumKernel(feature_map=feature_map, fidelity=fidelity)
            qsvc = QSVC(quantum_kernel=kernel)
            # QSVC uses standard 1D labels
            all_rows.append(self.train_evaluate(qsvc, "Quantum QSVC", size_name, q_data, is_vqc=False))

        self.results_df = pd.DataFrame(all_rows)
        return self.results_df

    def generate_plots(self):
        df = self.results_df
        if df.empty: return

        fig = plt.figure(figsize=(18, 12))
        gs = fig.add_gridspec(2, 2)

        # 1. Test Accuracy vs Data Size
        ax1 = fig.add_subplot(gs[0, 0])
        sns.lineplot(data=df, x='Data Size', y='Test Acc', hue='Model', marker='o', linewidth=2.5, ax=ax1)
        ax1.set_title('Test Accuracy vs Data Size', fontsize=14, fontweight='bold')
        ax1.set_ylabel('Accuracy')
        ax1.grid(True, alpha=0.3)

        # 2. Training Time vs Data Size
        ax2 = fig.add_subplot(gs[0, 1])
        sns.lineplot(data=df, x='Data Size', y='Time (s)', hue='Model', marker='s', linewidth=2.5, ax=ax2)
        ax2.set_title('Training Time Cost', fontsize=14, fontweight='bold')
        ax2.set_ylabel('Seconds')
        ax2.grid(True, alpha=0.3)

        # 3. Comprehensive Metrics (F1, Precision, Recall) - Bar Chart
        # Use the largest dataset available for this plot
        largest_size = list(self.data_sizes.keys())[-1]
        metrics_df = df[df['Data Size'] == largest_size].melt(
            id_vars=['Model'],
            value_vars=['Precision', 'Recall', 'F1 Score'],
            var_name='Metric', value_name='Score'
        )

        ax3 = fig.add_subplot(gs[1, 0])
        sns.barplot(data=metrics_df, x='Model', y='Score', hue='Metric', ax=ax3, palette='viridis')
        ax3.set_title(f'Detailed Metrics ({largest_size} Dataset)', fontsize=14, fontweight='bold')
        ax3.set_ylim(0, 1.0)
        ax3.legend(loc='lower right')

        # 4. Overfitting Analysis (Train vs Test Accuracy)
        acc_df = df[df['Data Size'] == largest_size].melt(
            id_vars=['Model'],
            value_vars=['Train Acc', 'Test Acc'],
            var_name='Type', value_name='Accuracy'
        )

        ax4 = fig.add_subplot(gs[1, 1])
        sns.barplot(data=acc_df, x='Model', y='Accuracy', hue='Type', ax=ax4, palette='RdBu')
        ax4.set_title(f'Overfitting Check: Train vs Test ({largest_size} Data)', fontsize=14, fontweight='bold')
        ax4.set_ylim(0, 1.1)

        plt.suptitle(f'Quantum vs Classical Sentiment Analysis (N Features={self.n_features})', fontsize=16)
        plt.tight_layout()
        plt.savefig('quantum_comparison_results.png')
        print("\nPlots saved to 'quantum_comparison_results.png'")
        plt.show()

    def print_text_report(self):
        print("\n" + "="*80)
        print("FINAL RESULTS TABLE")
        print("="*80)
        cols = ['Data Size', 'Model', 'Train Acc', 'Val Acc', 'Test Acc', 'Recall', 'F1 Score', 'Time (s)']
        print(self.results_df[cols].to_string(index=False, float_format="%.4f"))

if __name__ == "__main__":
    study = QuantumComparison(n_features=4)
    study.run()
    study.print_text_report()
    study.generate_plots()


... Processing Data (N=400) ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

sst2/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

sst2/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

sst2/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

  Training Classical LR... Done (0.01s)
  Training Classical SVC... Done (0.02s)
  Training Quantum VQC... 